# Bronze: ingesta con dlt (respaldo manual de Airflow)

Hace lo mismo que la tarea de Bronze del DAG: lee el Parquet de origen y lo deja en `bronze/<dataset>/<anio>/<dataset>_<anio>.parquet` (con override).

Reutiliza las funciones de `dags/scripts/bronze_ingest.py`, asi que Airflow y este notebook corren exactamente la misma logica.

**Antes de correr:** buckets `bronze`, `silver` y `gold` creados en MinIO y bloque `[bronze_ingest.destination.filesystem]` en `.dlt/secrets.toml`.

In [ ]:
import sys
from pathlib import Path

# Los scripts viven en dags/scripts (la misma carpeta que usa Airflow)
for candidato in [Path("../dags/scripts"), Path("/home/jovyan/dags/scripts"), Path("dags/scripts")]:
    if candidato.exists():
        sys.path.insert(0, str(candidato.resolve()))
        break

import dlt
import bronze_ingest as bronze

## Parametros

- Pipeline de Airflow: `users` 2021
- Cargas manuales: `users` 2020, `post` 2020, `post` 2021

In [ ]:
DATASET = "users"   # "users" o "post"
YEAR = 2021

## Resource: lee el Parquet por HTTP

Para `users` filtra por el anio de `CreationDate`; para `post` el origen ya viene separado por anio.

In [ ]:
@dlt.resource(name=DATASET, write_disposition="replace")
def my_df():
    # Lee el Parquet remoto por lotes (no carga el archivo completo en memoria)
    yield from bronze.read_batches(DATASET, YEAR)

## Pipeline: dlt -> MinIO (destino filesystem)

In [ ]:
pipeline = dlt.pipeline(
    pipeline_name=bronze.PIPELINE_NAME,
    destination="filesystem",
    dataset_name=bronze.STAGING_DATASET,
)

load_info = pipeline.run(
    my_df,
    loader_file_format="parquet",
)

In [ ]:
print(load_info)

## Mover el archivo a su ruta final de Bronze (override)

In [ ]:
ruta_final = bronze.move_to_final_path(pipeline, DATASET, YEAR)
print("Bronze listo:", ruta_final)

## Verificacion

In [ ]:
client = pipeline.destination_client()
fs = client.fs_client
fs.find(f"{client.bucket_path}/{DATASET}/{YEAR}")